In [4]:
import os
import json
import numpy as np
import polars as pl
from datetime import datetime

os.makedirs("statistics", exist_ok=True)

# Load cleaned data
print("Loading cleaned data...")
df = pl.read_parquet("cleanData/Final_Cleaned_1m.parquet")
print(f"Dataset shape: {df.shape}")
print(f"Unique services: {df['msname'].n_unique()}")
print(f"Timestamp range: {df['timestamp'].min()} to {df['timestamp'].max()}")

results = {}

# ==========================================
# STAT 1: Container CPU Utilization
# ==========================================
print("\nComputing Stat 1: CPU Utilization...")
percentiles = {}
for p in [50, 75, 90, 95, 99]:
    val = df['total_cpu_demand'].quantile(p/100)
    percentiles[f"p{p}"] = round(val, 4)
    print(f"  p{p:>3}: {val:.4f} cores")

total_services = df['msname'].n_unique()
low_cpu = df.filter(pl.col('total_cpu_demand') < 10)['msname'].n_unique()
p95 = df['total_cpu_demand'].quantile(0.95)
print(f"\n  Services with CPU < 10 cores: {low_cpu} out of {total_services} ({low_cpu/total_services*100:.1f}%)")
print(f"  → Proves flatline trap: 95th percentile = {p95:.4f} cores")

results['stat1_cpu_utilization'] = {
    "percentiles_cores": percentiles,
    "services_below_10_cores": int(low_cpu),
    "total_services": int(total_services),
    "pct_services_below_10_cores": round(low_cpu/total_services*100, 2),
    "p95_cores": round(p95, 4)
}

# ==========================================
# STAT 2: Node Stress vs Response Time
# ==========================================
print("\nComputing Stat 2: Node Stress vs RT...")
high_stress = df.filter(pl.col('avg_node_stress') > 0.40)
low_stress  = df.filter(pl.col('avg_node_stress') < 0.10)

high_rt_mean = high_stress['avg_response_time'].drop_nulls().mean()
low_rt_mean  = low_stress['avg_response_time'].drop_nulls().mean()
high_rt_p75  = high_stress['avg_response_time'].drop_nulls().quantile(0.75)
low_rt_p75   = low_stress['avg_response_time'].drop_nulls().quantile(0.75)
degradation  = (high_rt_mean - low_rt_mean) / low_rt_mean * 100 if low_rt_mean else 0

print(f"  High node stress (>40%):")
print(f"    Avg RT: {high_rt_mean:.4f} ms | p75 RT: {high_rt_p75:.4f} ms | Count: {len(high_stress):,}")
print(f"  Low node stress (<10%):")
print(f"    Avg RT: {low_rt_mean:.4f} ms | p75 RT: {low_rt_p75:.4f} ms | Count: {len(low_stress):,}")
print(f"  → RT degradation at high stress: {degradation:.1f}%")

results['stat2_node_stress_vs_rt'] = {
    "high_stress_above_40pct": {
        "sample_count": len(high_stress),
        "avg_response_time_ms": round(high_rt_mean, 4),
        "p75_response_time_ms": round(high_rt_p75, 4)
    },
    "low_stress_below_10pct": {
        "sample_count": len(low_stress),
        "avg_response_time_ms": round(low_rt_mean, 4),
        "p75_response_time_ms": round(low_rt_p75, 4)
    },
    "rt_degradation_pct": round(degradation, 2)
}

# ==========================================
# STAT 3: Workload Imbalance
# ==========================================
print("\nComputing Stat 3: Workload Imbalance...")
service_traffic = df.group_by("msname").agg(
    pl.col("total_traffic").sum().alias("lifetime_traffic")
).sort("lifetime_traffic", descending=True)

total_traffic  = df['total_traffic'].sum()
hotspot_results = {}

for pct in [0.05, 0.10, 0.20]:
    top_n = max(1, int(total_services * pct))
    top_traffic = service_traffic.head(top_n)['lifetime_traffic'].sum()
    share = round(top_traffic/total_traffic*100, 2)
    hotspot_results[f"top_{int(pct*100)}pct"] = {
        "service_count": top_n,
        "traffic_share_pct": share
    }
    print(f"  Top {pct*100:.0f}% ({top_n} services): {share}% of traffic")

results['stat3_workload_imbalance'] = {
    "total_services": int(total_services),
    "total_traffic_calls": float(total_traffic),
    "hotspot_analysis": hotspot_results
}

# ==========================================
# STAT 4: Communication Paradigm
# ==========================================
print("\nComputing Stat 4: Communication Paradigm...")
print("  Loading QPS data...")
df_qps = pl.read_parquet("cleanData/qps_pivoted_all.parquet")

mcr_cols = {
    "providerRPC_MCR": "RPC_Provider",
    "consumerRPC_MCR": "RPC_Consumer",
    "consumerMQ_MCR":  "Message_Queue",
    "HTTP_MCR":        "HTTP"
}

total_calls = 0
col_totals  = {}
for col, label in mcr_cols.items():
    if col in df_qps.columns:
        col_sum = df_qps[col].drop_nulls().sum()
        col_totals[label] = col_sum
        total_calls += col_sum

paradigm_breakdown = {}
print(f"  Total calls: {total_calls:,.0f}")
for label, val in col_totals.items():
    pct = round(val/total_calls*100, 2) if total_calls > 0 else 0
    paradigm_breakdown[label] = {
        "total_calls": float(val),
        "percentage": pct
    }
    print(f"  {label:<20}: {val:>15,.0f} calls ({pct:.1f}%)")

print(f"  → Compare to Alibaba paper: RPC=76%, MQ=23%, IP=1%")
results['stat4_communication_paradigm'] = {
    "total_calls": float(total_calls),
    "breakdown": paradigm_breakdown
}
del df_qps

# ==========================================
# STAT 5: Dataset Scale
# ==========================================
print("\nComputing Stat 5: Dataset Scale...")
duration_ms  = df['timestamp'].max() - df['timestamp'].min()
duration_min = duration_ms / 60000
duration_hrs = duration_min / 60

null_counts = {}
for col in df.columns:
    nulls = df[col].null_count()
    if nulls > 0:
        null_counts[col] = {
            "count": int(nulls),
            "percentage": round(nulls/df.shape[0]*100, 2)
        }

print(f"  Rows: {df.shape[0]:,} | Services: {total_services} | Duration: {duration_hrs:.1f} hrs")
results['stat5_dataset_scale'] = {
    "total_rows": df.shape[0],
    "total_columns": df.shape[1],
    "unique_services": int(total_services),
    "duration_minutes": round(duration_min, 0),
    "duration_hours": round(duration_hrs, 2),
    "null_counts": null_counts
}

# ==========================================
# STAT 6: Replica Count Distribution
# ==========================================
print("\nComputing Stat 6: Replica Distribution...")
replica_percentiles = {}
for p in [25, 50, 75, 90, 95, 99]:
    replica_percentiles[f"p{p}"] = int(df['replica_count'].quantile(p/100))
    print(f"  p{p:>3}: {replica_percentiles[f'p{p}']} replicas")

results['stat6_replica_distribution'] = {
    "min": int(df['replica_count'].min()),
    "max": int(df['replica_count'].max()),
    "mean": round(df['replica_count'].mean(), 2),
    "percentiles": replica_percentiles
}

# ==========================================
# STAT 7: Traffic Distribution
# ==========================================
print("\nComputing Stat 7: Traffic Distribution...")
traffic_percentiles = {}
for p in [50, 75, 90, 95, 99]:
    traffic_percentiles[f"p{p}"] = round(df['total_traffic'].quantile(p/100), 2)
    print(f"  p{p:>3}: {traffic_percentiles[f'p{p}']:,.2f} calls/min")

zero_traffic = df.filter(pl.col('total_traffic') == 0).shape[0]
results['stat7_traffic_distribution'] = {
    "mean": round(df['total_traffic'].mean(), 2),
    "max": round(df['total_traffic'].max(), 2),
    "zero_traffic_rows": int(zero_traffic),
    "zero_traffic_pct": round(zero_traffic/df.shape[0]*100, 2),
    "percentiles": traffic_percentiles
}

# ==========================================
# STAT 8: Node Stress Distribution
# ==========================================
print("\nComputing Stat 8: Node Stress Distribution...")
stress_percentiles = {}
for p in [25, 50, 75, 90, 95]:
    stress_percentiles[f"p{p}"] = round(df['avg_node_stress'].quantile(p/100), 4)
    print(f"  p{p:>3}: {stress_percentiles[f'p{p}']:.4f} ({stress_percentiles[f'p{p}']*100:.1f}%)")

high_stress_count = df.filter(pl.col('avg_node_stress') > 0.40).shape[0]
results['stat8_node_stress'] = {
    "mean": round(df['avg_node_stress'].mean(), 4),
    "mean_pct": round(df['avg_node_stress'].mean()*100, 2),
    "high_stress_rows": int(high_stress_count),
    "high_stress_pct": round(high_stress_count/df.shape[0]*100, 2),
    "percentiles": stress_percentiles
}

# ==========================================
# STAT 9: CPU Spike Analysis
# ==========================================
print("\nComputing Stat 9: CPU Spike Analysis...")
mean_cpu        = df['total_cpu_demand'].mean()
std_cpu         = df['total_cpu_demand'].std()
spike_threshold = mean_cpu + 2 * std_cpu
spikes          = df.filter(pl.col('total_cpu_demand') > spike_threshold)

print(f"  Mean: {mean_cpu:.4f} | Std: {std_cpu:.4f} | Threshold: {spike_threshold:.4f}")
print(f"  Spike events: {spikes.shape[0]:,} ({spikes.shape[0]/df.shape[0]*100:.2f}%)")
print(f"  Services with spikes: {spikes['msname'].n_unique():,}")

results['stat9_cpu_spikes'] = {
    "mean_cpu": round(mean_cpu, 4),
    "std_cpu": round(std_cpu, 4),
    "spike_threshold_2sigma": round(spike_threshold, 4),
    "spike_events": int(spikes.shape[0]),
    "spike_events_pct": round(spikes.shape[0]/df.shape[0]*100, 2),
    "services_with_spikes": int(spikes['msname'].n_unique())
}

# ==========================================
# STAT 10: Memory vs CPU Correlation
# ==========================================
print("\nComputing Stat 10: Memory-CPU Correlation...")
cpu    = df['total_cpu_demand'].to_numpy()
memory = df['total_memory_demand'].to_numpy()
corr   = np.corrcoef(cpu, memory)[0, 1]

interpretation = (
    "Strong — memory could be added as feature" if abs(corr) > 0.8
    else "Moderate — memory adds some signal" if abs(corr) > 0.5
    else "Weak — justified excluding memory from features"
)
print(f"  Correlation: {corr:.4f} → {interpretation}")

results['stat10_memory_cpu_correlation'] = {
    "correlation_coefficient": round(float(corr), 4),
    "interpretation": interpretation,
    "mean_memory": round(df['total_memory_demand'].mean(), 4),
    "max_memory": round(df['total_memory_demand'].max(), 4)
}

# ==========================================
# STAT 11: Response Time Analysis
# ==========================================
print("\nComputing Stat 11: Response Time Analysis...")
rt_data = df['avg_response_time'].drop_nulls()
rt_percentiles = {}
for p in [75, 90, 95, 99]:
    rt_percentiles[f"p{p}"] = round(rt_data.quantile(p/100), 4)
    print(f"  p{p:>3} RT: {rt_percentiles[f'p{p}']:.4f} ms")

results['stat11_response_time'] = {
    "non_null_rows": len(rt_data),
    "non_null_pct": round(len(rt_data)/df.shape[0]*100, 2),
    "mean_ms": round(rt_data.mean(), 4),
    "median_ms": round(rt_data.median(), 4),
    "max_ms": round(rt_data.max(), 4),
    "percentiles_ms": rt_percentiles
}

# ==========================================
# STAT 12: Per-Service CPU Variability
# ==========================================
print("\nComputing Stat 12: Per-Service CPU Variability...")
service_stats = df.group_by("msname").agg([
    pl.col("total_cpu_demand").mean().alias("mean_cpu"),
    pl.col("total_cpu_demand").std().alias("std_cpu"),
    pl.col("total_cpu_demand").max().alias("max_cpu"),
]).with_columns(
    (pl.col("std_cpu") / pl.col("mean_cpu")).alias("cv")
).sort("mean_cpu", descending=True)

top5_services = []
print(f"  {'Service':<20} {'Mean':>8} {'Std':>8} {'Max':>8} {'CV':>8}")
print(f"  {'-'*56}")
for row in service_stats.head(5).iter_rows(named=True):
    name = row['msname'][:18] + '..'
    print(f"  {name:<20} {row['mean_cpu']:>8.2f} {row['std_cpu']:>8.2f} "
          f"{row['max_cpu']:>8.2f} {row['cv']:>8.4f}")
    top5_services.append({
        "msname": row['msname'],
        "mean_cpu": round(row['mean_cpu'], 4),
        "std_cpu": round(row['std_cpu'], 4),
        "max_cpu": round(row['max_cpu'], 4),
        "cv": round(row['cv'], 4)
    })

results['stat12_per_service_variability'] = {
    "median_cv": round(float(service_stats['cv'].median()), 4),
    "top5_highest_cpu_services": top5_services
}

# ==========================================
# Save all results
# ==========================================
print("\n" + "="*50)
print("Saving all results...")

timestamp  = datetime.now().strftime("%Y%m%d_%H%M%S")
json_path  = f"statistics/dataset_statistics_{timestamp}.json"
txt_path   = f"statistics/dataset_statistics_{timestamp}.txt"

# Save JSON
with open(json_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"✅ JSON saved: {json_path}")

# Save TXT summary
with open(txt_path, "w") as f:
    f.write("ALIBABA MICROSERVICE DATASET — STATISTICAL SUMMARY\n")
    f.write(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"Dataset:   {df.shape[0]:,} rows × {df.shape[1]} columns\n")
    f.write(f"Services:  {total_services}\n")
    f.write("="*60 + "\n\n")

    for stat_name, stat_data in results.items():
        f.write(f"{stat_name.upper()}\n")
        f.write("-"*40 + "\n")
        f.write(json.dumps(stat_data, indent=2))
        f.write("\n\n")

print(f"✅ TXT saved: {txt_path}")
print(f"\nAll statistics saved to 'statistics/' folder:")
print(f"  {json_path}")
print(f"  {txt_path}")

Loading cleaned data...
Dataset shape: (937986, 8)
Unique services: 1303
Timestamp range: 0 to 43140000

Computing Stat 1: CPU Utilization...
  p 50: 2.2968 cores
  p 75: 8.5862 cores
  p 90: 31.1733 cores
  p 95: 58.4401 cores
  p 99: 182.8870 cores

  Services with CPU < 10 cores: 1058 out of 1303 (81.2%)
  → Proves flatline trap: 95th percentile = 58.4401 cores

Computing Stat 2: Node Stress vs RT...
  High node stress (>40%):
    Avg RT: 36.3413 ms | p75 RT: 34.5969 ms | Count: 920,301
  Low node stress (<10%):
    Avg RT: 138.4931 ms | p75 RT: 24.0039 ms | Count: 6,619
  → RT degradation at high stress: -73.8%

Computing Stat 3: Workload Imbalance...
  Top 5% (65 services): 62.59% of traffic
  Top 10% (130 services): 77.33% of traffic
  Top 20% (260 services): 89.44% of traffic

Computing Stat 4: Communication Paradigm...
  Loading QPS data...
  Total calls: 13,517,808,697
  RPC_Provider        :   6,136,091,925 calls (45.4%)
  RPC_Consumer        :   4,976,035,915 calls (36.8%)
 